# Breakthrough stage: generator structure, kernel AUC ranking, learned RBF metric


In [1]:
from pathlib import Path
from time import perf_counter
import random, numpy as np, pandas as pd
from IPython.display import display
from exp_3_experiments import *
SEED=0xFACED; random.seed(SEED); np.random.seed(SEED)


## Data and leakage guards


In [2]:
DATA_DIR=Path('./data') if not Path('/kaggle/input').exists() else Path('/kaggle/input/competitions/week-03-hidden-pairs')
train_df=pd.read_csv(DATA_DIR/'train.csv'); val_df=pd.read_csv(DATA_DIR/'validation.csv')
features=[c for c in train_df if c not in {'row_id','target'}]; assert 'row_id' not in features and len(features)==512
X_train=train_df[features].to_numpy(); y_train=train_df.target.to_numpy(); X_val=val_df[features].to_numpy(); y_val=val_df.target.to_numpy()
assert np.isfinite(X_train).all() and (X_train>=0).all()


## Configuration


In [3]:
N_SPLITS=5; CV_SEEDS=[SEED+i for i in range(5)]; N_JOBS=-1
RUN_REVERSE_ENGINEERING=True
RUN_RAW_AUC_RANKER=True; RUN_CURRENT_AUC_RANKER=True; RUN_HARD_PAIR_RANKER=True
RUN_DIAGONAL_METRIC=True; RUN_LOWRANK_METRIC=True; RUN_JOINT_METRIC_RANKER=False
RAW_C=3.447606; RAW_GAMMA=.003110
CURRENT_CONFIG='coord_monotonic'
CURRENT_PARAMS={'kernel_type':'product','C':18.152351,'gamma_raw':.002496,'gamma_geom':.010154}
records={}


# Phase A — reverse engineering


In [4]:
if RUN_REVERSE_ENGINEERING:
    duplicate_summary=duplicate_diagnostics(X_train,y_train)
    nearest_rows,cross_validation_nn=nearest_diagnostics(X_train,y_train,X_val)
    zero_summary=zero_pattern_diagnostics(X_train,y_train)
    display(pd.DataFrame([duplicate_summary])); display(nearest_rows.groupby('same_target').agg(['mean','median']))
    display(pd.DataFrame([zero_summary])); print('train→validation nearest distances',pd.Series(cross_validation_nn).describe())


,n_duplicate_groups,max_group_size,rows_in_duplicates,consistent_groups
0,0,1,0,0


euclidean               cosine          
                  mean     median      mean    median
same_target                                          
False        23.859593  23.950353  0.566194  0.590475
True         19.841355  21.093594  0.438030  0.491676

,repeated_masks,largest_group,rows_repeated,median_nn_hamming
0,114,784,1299,0.0


train→validation nearest distances count    560.000000
mean      21.374691
std        6.842069
min        0.599677
25%       17.827462
50%       22.309398
75%       26.583569
max       35.012405
dtype: float64


In [5]:
if RUN_REVERSE_ENGINEERING:
    triangle_relations=triangle_search(X_train,n_triples=500_000,seed=SEED)
    best_power=float(triangle_relations.iloc[0].power); null_calibration=triangle_null_calibration(X_train,best_power,20_000,SEED)
    display(triangle_relations.head(50)); display(pd.DataFrame(null_calibration,index=['q0.001','q0.01','q0.05']).T)
    null_floor=min(v[0] for v in null_calibration.values())
    strong_triangle=triangle_relations.iloc[0].triangle_score < null_floor*0.1
    print('STRONG TRIANGLE STRUCTURE DETECTED' if strong_triangle else 'NO EVIDENCE FOR TRIANGLE/DISTANCE GENERATOR')
    if strong_triangle:
        top=triangle_relations.head(100).copy(); top['target_pattern']=[tuple(y_train[[r.row_i,r.row_j,r.row_k]]) for _,r in top.iterrows()]
        display(top.target_pattern.value_counts())
        combined=np.vstack([X_train,X_val]); cross_relations=triangle_search(combined,powers=(best_power,),n_triples=200_000,seed=SEED+99)
        strong_cross=cross_relations[cross_relations.triangle_score < null_floor*0.1]
        validation_ids=set(range(len(X_train),len(combined))); covered=set(strong_cross.row_i)|set(strong_cross.row_j)|set(strong_cross.row_k)
        validation_covered=len(covered & validation_ids); print('validation structural coverage:',validation_covered,len(X_val),validation_covered/len(X_val))
    else:
        validation_covered=0


,row_i,row_j,row_k,power,triangle_score,q90,q99,fraction_good_coordinates
0,829,731,956,0.50,0.073773,0.280923,0.635905,0.001953
1,2065,956,853,0.50,0.096633,0.241876,0.500845,0.000000
2,621,2139,2089,1.00,0.098342,0.349263,0.852854,0.001953
3,2116,731,1083,0.50,0.099388,0.367433,0.575955,0.000000
4,924,1260,1211,0.50,0.101361,0.323930,0.604651,0.000000
5,626,924,588,0.50,0.103303,0.280473,0.630145,0.000000
6,218,502,1084,0.75,0.104024,0.366202,0.826451,0.003906
7,678,2284,1457,0.50,0.106589,0.288807,0.537041,0.000000
8,1270,1151,626,0.50,0.107357,0.285203,0.602439,0.000000
9,2195,1992,1740,0.50,0.107573,0.404878,0.678904,0.000000


,q0.001,q0.01,q0.05
random_triples,0.110734,0.116657,0.121525
within_row_permutation,0.113517,0.118031,0.122641
column_shuffle,0.115252,0.119909,0.124597


NO EVIDENCE FOR TRIANGLE/DISTANCE GENERATOR


# Baselines on identical 25 folds


In [6]:
raw_cache=raw_kernel_cache(X_train,y_train,CV_SEEDS,N_SPLITS,gamma=RAW_GAMMA,n_jobs=N_JOBS)
raw_aucs,raw_oof=svc_scores(raw_cache,RAW_C,N_JOBS)
raw_val,raw_val_score=raw_svc_validation(X_train,y_train,X_val,y_val,RAW_C,RAW_GAMMA)
current_cache=current_kernel_cache(X_train,y_train,CV_SEEDS,CURRENT_CONFIG,CURRENT_PARAMS,N_SPLITS,n_jobs=1)
current_aucs,current_oof=svc_scores(current_cache,CURRENT_PARAMS['C'],N_JOBS)
from svc_new_experiments import current_best_validation
current_val,current_val_score=current_best_validation(X_train,y_train,X_val,y_val,CURRENT_CONFIG,CURRENT_PARAMS,SEED)


# Phase B — Kernel AUC Ranker


In [7]:
if RUN_RAW_AUC_RANKER:
    t=perf_counter(); a,o,p=ranker_scores(raw_cache,n_jobs=N_JOBS); v,vs=fit_external_ranker(X_train,y_train,X_val,y_val,'raw',{'gamma':RAW_GAMMA},{**p,'max_iter':150})
    records['B1 Raw RBF AUC Ranker']=(a,o,v,perf_counter()-t,p)
if RUN_CURRENT_AUC_RANKER:
    t=perf_counter(); a,o,p=ranker_scores(current_cache,n_jobs=N_JOBS); v,vs=fit_external_ranker(X_train,y_train,X_val,y_val,'current',CURRENT_PARAMS,{**p,'max_iter':150},CURRENT_CONFIG,SEED)
    records['B2 Current-Best Kernel AUC Ranker']=(a,o,v,perf_counter()-t,p)
if RUN_HARD_PAIR_RANKER and RUN_CURRENT_AUC_RANKER and records['B2 Current-Best Kernel AUC Ranker'][0].mean()>=current_aucs.mean()-0.001:
    for frac in (.25,.5):
        t=perf_counter(); a,o,p=ranker_scores(current_cache,hard_fraction=frac,n_jobs=N_JOBS); v,vs=fit_external_ranker(X_train,y_train,X_val,y_val,'current',CURRENT_PARAMS,{**p,'max_iter':150},CURRENT_CONFIG,SEED)
        records[f'B3 Hard-Pair Ranker {frac}']=(a,o,v,perf_counter()-t,p)


# Phase C — learned RBF metric


In [8]:
metric_models={}
if RUN_DIAGONAL_METRIC:
    best=None
    for reg in (1e-3,1e-2,1e-1,1.):
        t=perf_counter(); a,o,mods=learned_metric_scores(X_train,y_train,CV_SEEDS,0,reg,n_splits=N_SPLITS,n_jobs=N_JOBS,seed=SEED)
        if best is None or a.mean()>best[0].mean(): best=(a,o,mods,reg,perf_counter()-t)
    a,o,mods,reg,tm=best; v,vs,m=learned_metric_validation(X_train,y_train,X_val,y_val,0,reg,seed=SEED)
    records['C1 Learned diagonal RBF metric']=(a,o,v,tm,{'rank':0,'lambda_identity':reg}); metric_models[0]=m


In [9]:
if RUN_LOWRANK_METRIC and RUN_DIAGONAL_METRIC and records['C1 Learned diagonal RBF metric'][0].mean()>=raw_aucs.mean()-0.003:
    for rank in (4,8,16):
        best=None
        for ri in (1e-2,1e-1):
            for rl in (1e-2,1e-1):
                t=perf_counter(); a,o,mods=learned_metric_scores(X_train,y_train,CV_SEEDS,rank,ri,rl,N_SPLITS,N_JOBS,SEED)
                if best is None or a.mean()>best[0].mean(): best=(a,o,mods,ri,rl,perf_counter()-t)
        a,o,mods,ri,rl,tm=best; v,vs,m=learned_metric_validation(X_train,y_train,X_val,y_val,rank,ri,rl,SEED)
        records[f'C Learned metric rank={rank}']=(a,o,v,tm,{'rank':rank,'lambda_identity':ri,'lambda_lowrank':rl}); metric_models[rank]=m


# Paired comparison and metric inspection


In [10]:
rows=[]
rows.append(summary_row('Raw RBF SVC','Baseline',raw_aucs,raw_oof,raw_val,raw_aucs,current_aucs,current_oof,0,{'C':RAW_C,'gamma':RAW_GAMMA}))
rows.append(summary_row('Current Best SVC','Baseline',current_aucs,current_oof,current_val,raw_aucs,current_aucs,current_oof,0,CURRENT_PARAMS))
for name,(a,o,v,t,p) in records.items(): rows.append(summary_row(name,'Ranker' if name.startswith('B') else 'Learned metric',a,o,v,raw_aucs,current_aucs,current_oof,t,p))
results=pd.DataFrame(rows).sort_values('CV_mean',ascending=False); display(results.drop(columns=['oof']))
if metric_models:
    for rank,m in metric_models.items():
        print('rank',rank,'weights',pd.Series(m.weights_).describe(percentiles=[.1,.5,.9]).to_dict(),'gradient',m.gradient_norm_)
        if m.U_array_ is not None: print('singular values',np.linalg.svd(m.U_array_,compute_uv=False)[:10])


,experiment,family,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_current,losses_vs_current,wilcoxon_vs_current,validation_auc,corr_with_current,fit_time,params,auc_per_fold
1,Current Best SVC,Baseline,0.931089,0.006840,0.930099,0.002243,0.000000,0,0,1.000000e+00,0.953571,1.000000,0.000000,"{'kernel_type': 'product', 'C': 18.152351, 'ga...","[0.925059838750315, 0.9395786092214664, 0.9309..."
0,Raw RBF SVC,Baseline,0.928846,0.007416,0.927674,0.000000,-0.002243,6,19,2.498031e-04,0.947028,0.985965,0.000000,"{'C': 3.447606, 'gamma': 0.00311}","[0.9208238851095993, 0.9342088687326783, 0.927..."
4,C1 Learned diagonal RBF metric,Learned metric,0.926805,0.007907,0.924808,-0.002041,-0.004284,2,23,1.490116e-06,0.947117,0.984596,47.338490,"{'rank': 0, 'lambda_identity': 1.0}","[0.9248078861174099, 0.9318625598387502, 0.925..."
3,B2 Current-Best Kernel AUC Ranker,Ranker,0.925865,0.009312,0.924808,-0.002981,-0.005224,3,22,6.446058e-05,0.940217,0.978419,364.536954,"{'lambda_reg': 0.0001, 'temperature': 1.0, 'ha...","[0.9093442932728647, 0.9384920634920634, 0.927..."
2,B1 Raw RBF AUC Ranker,Ranker,0.909199,0.010784,0.907707,-0.019647,-0.021890,0,25,5.960464e-08,0.918750,0.939653,404.229505,"{'lambda_reg': 0.0001, 'temperature': 1.0, 'ha...","[0.8869992441421013, 0.9253747795414462, 0.907..."
5,C Learned metric rank=4,Learned metric,0.903426,0.009258,0.900022,-0.025420,-0.027663,0,25,5.960464e-08,0.920332,0.921574,56.851588,"{'rank': 4, 'lambda_identity': 0.1, 'lambda_lo...","[0.8991717057193248, 0.8999433106575963, 0.917..."
6,C Learned metric rank=8,Learned metric,0.902272,0.008669,0.899077,-0.026574,-0.028817,0,25,5.960464e-08,0.919273,0.918185,53.723003,"{'rank': 8, 'lambda_identity': 0.1, 'lambda_lo...","[0.8990772234819854, 0.8967466616276141, 0.913..."
7,C Learned metric rank=16,Learned metric,0.900734,0.009770,0.899550,-0.028112,-0.030355,0,25,5.960464e-08,0.914031,0.915058,53.791168,"{'rank': 16, 'lambda_identity': 0.1, 'lambda_l...","[0.8995496346686823, 0.8953136810279669, 0.911..."


rank 0 weights {'count': 512.0, 'mean': 0.9753150939941406, 'std': 0.05255671218037605, 'min': 0.8616654276847839, '10%': 0.9208618402481079, '50%': 0.9668669998645782, '90%': 1.0371195197105407, 'max': 1.2358673810958862} gradient 4.920592255075462e-05
rank 4 weights {'count': 512.0, 'mean': 0.5158369541168213, 'std': 0.03215129300951958, 'min': 0.4162316620349884, '10%': 0.47496547996997834, '50%': 0.5150732100009918, '90%': 0.5551021575927735, 'max': 0.6031047701835632} gradient 0.00013370956003200263
singular values [18.823885   4.1512203  1.2757416  0.5953663]
rank 8 weights {'count': 512.0, 'mean': 0.5261231064796448, 'std': 0.03212204948067665, 'min': 0.4242035746574402, '10%': 0.4850521951913834, '50%': 0.5258808135986328, '90%': 0.5643301486968995, 'max': 0.6137597560882568} gradient 0.00011954651563428342
singular values [21.414093    4.037469    3.872206    1.941837    1.5476027   1.2172264
  0.90466595  0.5719293 ]
rank 16 weights {'count': 512.0, 'mean': 0.533608615398407,

# Breakthrough summary


In [11]:
best=results.iloc[0]; breakthrough=results[(results.delta_vs_current>=.002)&(results.wins_vs_current>=18)&(results.validation_auc>=current_val-.001)]
print('BREAKTHROUGH STAGE SUMMARY')
print('Phase A:', 'STRONG TRIANGLE STRUCTURE' if RUN_REVERSE_ENGINEERING and strong_triangle else 'NEGATIVE')
print('Phase B Raw/Current SVC:',raw_aucs.mean(),current_aucs.mean())
print('Overall best:',best.experiment,'delta',best.delta_vs_current,'wins',best.wins_vs_current)
print('BREAKTHROUGH CANDIDATES' if len(breakthrough) else 'NO BREAKTHROUGH CANDIDATE'); display(breakthrough.drop(columns=['oof']))
print('No test access or submission.')


BREAKTHROUGH STAGE SUMMARY
Phase A: NEGATIVE
Phase B Raw/Current SVC: 0.9288460569412949 0.9310890652557318
Overall best: Current Best SVC delta 0.0 wins 0
NO BREAKTHROUGH CANDIDATE


,experiment,family,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_current,losses_vs_current,wilcoxon_vs_current,validation_auc,corr_with_current,fit_time,params,auc_per_fold


No test access or submission.
